## Tool definition

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [5]:
tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number.",
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Approximately 21.61018278497431 (the square root of 467 is irrational). Rounded: 21.61018.


In [22]:
from pprint import pprint

def pretty_print_messages(messages):
    for i, msg in enumerate(messages):
        print(f"\n--- Message {i+1} ---")

        print(f"Type: {msg.__class__.__name__}")
        print(f"ID: {msg.id}")

        if hasattr(msg, "content") and msg.content:
            print(f"Content: {msg.content}")

        if hasattr(msg, "tool_calls") and msg.tool_calls:
            print("Tool Calls:")
            for tc in msg.tool_calls:
                print(f"  - name: {tc['name']}")
                print(f"    args: {tc['args']}")

        if hasattr(msg, "name"):
            print(f"Tool Name: {msg.name}")

        if hasattr(msg, "usage_metadata") and msg.usage_metadata:
            print("Usage:")
            print(msg.usage_metadata)

pretty_print_messages(response['messages'])

# pprint(response['messages'], depth=3)



--- Message 1 ---
Type: HumanMessage
ID: e67adb26-f134-4436-92fe-9ab8bfb4417b
Content: What is the square root of 467?
Tool Name: None

--- Message 2 ---
Type: AIMessage
ID: lc_run--019db805-e682-7e41-9140-f998376b3cae-0
Tool Calls:
  - name: square_root
    args: {'x': 467}
Tool Name: None
Usage:
{'input_tokens': 158, 'output_tokens': 727, 'total_tokens': 885, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 704}}

--- Message 3 ---
Type: ToolMessage
ID: 468d513c-6b68-4ac3-a41f-bd8070167c52
Content: 21.61018278497431
Tool Name: square_root

--- Message 4 ---
Type: AIMessage
ID: lc_run--019db805-ff75-7052-8127-51c9664975ec-0
Content: Approximately 21.61018278497431 (the square root of 467 is irrational). Rounded: 21.61018.
Tool Name: None
Usage:
{'input_tokens': 193, 'output_tokens': 484, 'total_tokens': 677, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 448}}


In [23]:
import json

def to_serializable(messages):
    return [m.dict() for m in messages]

print(json.dumps(to_serializable(response['messages']), indent=2))

[
  {
    "content": "What is the square root of 467?",
    "additional_kwargs": {},
    "response_metadata": {},
    "type": "human",
    "name": null,
    "id": "e67adb26-f134-4436-92fe-9ab8bfb4417b"
  },
  {
    "content": "",
    "additional_kwargs": {
      "refusal": null
    },
    "response_metadata": {
      "token_usage": {
        "completion_tokens": 727,
        "prompt_tokens": 158,
        "total_tokens": 885,
        "completion_tokens_details": {
          "accepted_prediction_tokens": 0,
          "audio_tokens": 0,
          "reasoning_tokens": 704,
          "rejected_prediction_tokens": 0
        },
        "prompt_tokens_details": {
          "audio_tokens": 0,
          "cached_tokens": 0
        }
      },
      "model_provider": "openai",
      "model_name": "gpt-5-nano-2025-08-07",
      "system_fingerprint": null,
      "id": "chatcmpl-DXdSjNdbTQ75CNCkhj0gud8LPDKa9",
      "service_tier": "default",
      "finish_reason": "tool_calls",
      "logprobs": null


/var/folders/xx/ypnl5f2n0y7b48w_pgxyhqt80000gn/T/ipykernel_29112/689695730.py:4: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return [m.dict() for m in messages]


In [13]:
import json

print(json.dumps([m.model_dump() for m in response["messages"]], indent=2))

[
  {
    "content": "What is the square root of 467?",
    "additional_kwargs": {},
    "response_metadata": {},
    "type": "human",
    "name": null,
    "id": "e67adb26-f134-4436-92fe-9ab8bfb4417b"
  },
  {
    "content": "",
    "additional_kwargs": {
      "refusal": null
    },
    "response_metadata": {
      "token_usage": {
        "completion_tokens": 727,
        "prompt_tokens": 158,
        "total_tokens": 885,
        "completion_tokens_details": {
          "accepted_prediction_tokens": 0,
          "audio_tokens": 0,
          "reasoning_tokens": 704,
          "rejected_prediction_tokens": 0
        },
        "prompt_tokens_details": {
          "audio_tokens": 0,
          "cached_tokens": 0
        }
      },
      "model_provider": "openai",
      "model_name": "gpt-5-nano-2025-08-07",
      "system_fingerprint": null,
      "id": "chatcmpl-DXdSjNdbTQ75CNCkhj0gud8LPDKa9",
      "service_tier": "default",
      "finish_reason": "tool_calls",
      "logprobs": null


In [14]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_S5Y40hRpOhxsCdg5SVa5yAvd', 'type': 'tool_call'}]


In [32]:
import sys
sys.path.append('/Users/bohaoli/Desktop/tuto/tuto_langchain/official/001-foundation-langraph')
import importlib
import printer
importlib.reload(printer)
from printer import display_langchain_messages_dark  # noqa: E402
display_langchain_messages_dark(response["messages"])